In [1]:
import os
os.environ["USE_TF"] = "0"  
os.environ["USE_KERAS"] = "0"
from dotenv import load_dotenv
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")

In [2]:
import os
import psycopg2
import lancedb
from lancedb.pydantic import Vector, LanceModel
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import numpy as np
import time

# --- КОНФИГУРАЦИЯ МОДЕЛИ ---
print("[DEBUG] Загрузка модели SentenceTransformer (device='mps')...")
model = SentenceTransformer('all-MiniLM-L6-v2', device='mps') 
print("[DEBUG] Модель загружена.")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=150
)

class ArxivChunk(LanceModel):
    vector: Vector(384)
    id: str
    article_id: str
    title: str
    text: str

db_params = {
    "dbname": "arxivdb",
    "user": "abakumov",
    "password": "obdFmjNDLUve",
    "host": "82.146.52.52",
    "port": "5433"
}

def migrate(limit_total=100000, batch_size=150):
    checkpoint_file = "last_id.txt"
    last_id = "9999.99999" 
    
    # 1. Загрузка чекпоинта
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, "r") as f:
            last_id = f.read().strip()
        print(f"[DEBUG] Чекпоинт найден. Начинаем с ID: {last_id}")
    else:
        print(f"[DEBUG] Чекпоинт не найден. Начинаем с нуля (ID: {last_id})")

    # 2. Подключение к LanceDB
    print("[DEBUG] Подключение к LanceDB...")
    db = lancedb.connect("./arxiv_lancedb")
    
    if "papers" in db.table_names():
        print("[DEBUG] Таблица 'papers' уже существует. Открываю...")
        table = db.open_table("papers")
    else:
        print("[DEBUG] Таблица 'papers' не найдена. Создаю новую...")
        table = db.create_table("papers", schema=ArxivChunk)

    total_processed_articles = 0
    
    # 3. Основной цикл
    while total_processed_articles < limit_total:
        try:
            print(f"\n[DEBUG] --- Запрос к PostgreSQL (найти {batch_size} статей < {last_id}) ---")
            start_time = time.time()
            
            with psycopg2.connect(**db_params) as conn:
                with conn.cursor() as cursor:
                    cursor.execute("""
                        SELECT id, title, clean_text 
                        FROM arxivdb.public.articles 
                        WHERE id < %s 
                        AND clean_text IS NOT NULL 
                        AND section_text_new IS NOT NULL
                        ORDER BY id DESC 
                        LIMIT %s
                    """, (last_id, batch_size))
                    rows = cursor.fetchall()
            
            sql_time = time.time() - start_time
            if not rows:
                print("[DEBUG] SQL вернул 0 строк. Данные в базе закончились.")
                break
            
            print(f"[DEBUG] SQL запрос выполнен за {sql_time:.2f} сек. Получено статей: {len(rows)}")

            data_to_insert = []
            
            for row_idx, row in enumerate(rows):
                article_id = str(row[0]) 
                title, text = row[1], row[2]
                
                if not text or len(text) < 50: 
                    print(f"[DEBUG]   Статья {article_id} слишком короткая. Пропуск.")
                    last_id = article_id
                    continue

                # Разбиение на чанки
                chunks = splitter.split_text(text)
                
                # Генерация эмбеддингов (самый опасный момент для MPS)
                print(f"[DEBUG]   [{row_idx+1}/{len(rows)}] Обработка {article_id}: {len(chunks)} чанков...", end="\r")
                
                try:
                    embeddings = model.encode(chunks, show_progress_bar=False)
                except Exception as e:
                    print(f"\n[ERROR] Ошибка генерации эмбеддингов для {article_id}: {e}")
                    # Если зависло на MPS, попробуйте сменить на 'cpu' в начале скрипта
                    continue

                for i, (chunk_text, emb) in enumerate(zip(chunks, embeddings)):
                    data_to_insert.append({
                        "vector": emb,
                        "id": f"{article_id}_{i}",
                        "article_id": article_id,
                        "title": title,
                        "text": chunk_text
                    })
                
                last_id = article_id 

            # 4. Вставка в LanceDB
            if data_to_insert:
                print(f"\n[DEBUG] Запись в LanceDB {len(data_to_insert)} чанков...")
                table.add(data_to_insert)
                
                total_processed_articles += len(rows)
                
                # Обновление чекпоинта
                with open(checkpoint_file, "w") as f:
                    f.write(last_id)
                
                print(f"[DEBUG] Успешно сохранено. Всего статей: {total_processed_articles}. Последний ID: {last_id}")
            else:
                print("\n[DEBUG] Нечего вставлять в этом батче.")
                # Если батч был пустой (все статьи < 50 символов), сохраняем last_id, чтобы не зациклиться
                with open(checkpoint_file, "w") as f:
                    f.write(last_id)

        except Exception as e:
            print(f"\n[CRITICAL ERROR] Ошибка во время миграции: {e}")
            print("[DEBUG] Сплю 5 секунд перед повтором...")
            time.sleep(5)
            continue

    print(f"\n[DEBUG] Миграция завершена! Итого статей: {total_processed_articles}")
    
    print("[DEBUG] Начинаю создание FTS индекса (это может занять время)...")
    table.create_fts_index("text", replace=True)
    print("[DEBUG] Индекс создан.")

# Запуск
if __name__ == "__main__":
    migrate()

/Library/Frameworks/Python.framework/Versions/3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[DEBUG] Загрузка модели SentenceTransformer (device='mps')...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15446.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[DEBUG] Модель загружена.
[DEBUG] Чекпоинт найден. Начинаем с ID: 2602.12246
[DEBUG] Подключение к LanceDB...
[DEBUG] Таблица 'papers' уже существует. Открываю...

[DEBUG] --- Запрос к PostgreSQL (найти 150 статей < 2602.12246) ---


/var/folders/5w/td1qssr102g_shnry7z0q4k00000gn/T/ipykernel_80502/3813776070.py:52: DeprecationWarning: table_names() is deprecated, use list_tables() instead
  if "papers" in db.table_names():


[DEBUG] SQL запрос выполнен за 0.25 сек. Получено статей: 150
[DEBUG]   [150/150] Обработка 2602.11945: 33 чанков....
[DEBUG] Запись в LanceDB 5116 чанков...
[DEBUG] Успешно сохранено. Всего статей: 150. Последний ID: 2602.11945

[DEBUG] --- Запрос к PostgreSQL (найти 150 статей < 2602.11945) ---
[DEBUG] SQL запрос выполнен за 0.80 сек. Получено статей: 150
[DEBUG]   [150/150] Обработка 2602.11691: 96 чанков....
[DEBUG] Запись в LanceDB 4684 чанков...
[DEBUG] Успешно сохранено. Всего статей: 300. Последний ID: 2602.11691

[DEBUG] --- Запрос к PostgreSQL (найти 150 статей < 2602.11691) ---
[DEBUG] SQL запрос выполнен за 0.25 сек. Получено статей: 150
[DEBUG]   [150/150] Обработка 2602.11439: 65 чанков...
[DEBUG] Запись в LanceDB 5073 чанков...
[DEBUG] Успешно сохранено. Всего статей: 450. Последний ID: 2602.11439

[DEBUG] --- Запрос к PostgreSQL (найти 150 статей < 2602.11439) ---
[DEBUG] SQL запрос выполнен за 0.75 сек. Получено статей: 150
[DEBUG]   [150/150] Обработка 2602.11166: 18 

KeyboardInterrupt: 

In [15]:
import lancedb
from sentence_transformers import SentenceTransformer

db = lancedb.connect("./arxiv_lancedb")
table = db.open_table("papers")
model = SentenceTransformer('all-MiniLM-L6-v2', device='mps')

def search(query: str, limit: int = 5):
    query_vector = model.encode(query)
    results = table.search(query_vector).limit(limit).to_pandas()
    return results

results = search("How do transformers work in computer vision?")
display(results)

,vector,id,article_id,title,text,_distance
0,"[-0.08818832, 0.026633421, -0.016104897, -0.07...",1204.2139_2,1204.2139,affine image registration transformation estim...,of the transformation. IR methods can also be ...,0.979784
1,"[-0.10935743, 0.05847293, 0.023091929, -0.0960...",1005.5181_3,1005.5181,compression rate method for empirical science ...,"it considers, the field can realize enormous m...",0.990906
2,"[-0.060275495, 0.01198954, -0.044333097, -0.06...",1107.0845_1,1107.0845,automatic road lighting system (arls) model ba...,road lighting system (ARLS) consists of four m...,0.993459
3,"[-0.074987166, 0.08672547, -0.011009323, -0.05...",1104.5466_117,1104.5466,notes on a new philosophy of empirical science,can be generalized over a wide range of applic...,1.004861
4,"[-0.0828432, -0.02273637, -0.000596193, -0.012...",0908.3359_0,0908.3359,geometric analysis of the conformal camera for...,"Introduction In the last few years, we have de...",1.009124


In [17]:
print(results['text'].iloc[1])

it considers, the field can realize enormous methodological advantages. A crucial argument, contained in Section 3.4, is that the shift from the traditional formulation of vision to the vision-as-compression approach is not really very drastic. Nearly all computer vision tasks can be easily reformulated as special techniques for image compression. For example, image segmentation can be viewed as a special way of compressing an image by separating the pixels into homogeneous regions, so that bits can be saved by encoding the pixels with a region-specific model . The stereo matching problem can be reformulated as a special way of compressing a stereo pair by using the first image, plus a disparity function, to predict the pixels in the second image . To make the ideas of this paper as concrete as possible, Section 3.2 gives one simple proposal. Here, a camera is set up next to a highway, and is used to obtain a large video database. Assuming the background is mostly static, the major sou